# Seizure Signal Visualization & Detection Analysis

Comprehensive analysis of normal vs seizure ECG/EEG signals with model predictions and detection performance visualization using Weights & Biases logging.

**Notebook Sections:**
1. Configuration and setup
2. Load SeizeIT2 dataset
3. Signal preprocessing
4. Signal visualization (normal vs seizure)
5. Model inference on test signals
6. Prediction visualization
7. Detection performance analysis
8. Export results to wandb

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import logging
from typing import Dict, Tuple

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# PyTorch
import torch
from torch.utils.data import DataLoader

# Weights & Biases
try:
    import wandb
    HAS_WANDB = True
    print("✓ wandb available")
except ImportError:
    HAS_WANDB = False
    print("✗ wandb not available")

# Project imports
from config.config import DEFAULT_CONFIG
from src.data_loader import BIDSDataLoader
from src.visualization import SignalVisualizer
from src.models import ModelFactory

print(f"✓ Project root: {project_root}")
print(f"✓ Config loaded")

## 1. Load SeizeIT2 Dataset

Load the BIDS-formatted SeizeIT2 dataset and inspect available recordings.

In [ ]:
# Load BIDS dataset
loader = BIDSDataLoader(DEFAULT_CONFIG.data)

# List all recordings
recordings = loader.list_all_recordings()
total_seizures = sum(r['num_seizures'] for r in recordings)

print(f"Dataset loaded successfully!")
print(f"Total subjects: {len(loader.get_subjects())}")
print(f"Total recordings: {len(recordings)}")
print(f"Total seizures found: {total_seizures}")

# Show sample recordings
print("\nSample recordings:")
for i, rec in enumerate(recordings[:5]):
    print(f"  {i+1}. {rec['subject_id']}/{rec['session_id']}/run-{rec['run_id']:02d}: {rec['num_seizures']} seizures")

## 2. Create Visualization Examples

Load example signals and create comparison plots of normal vs seizure recordings.

In [ ]:
# Find recordings with and without seizures
with_seizures = [r for r in recordings if r['num_seizures'] > 0]
without_seizures = [r for r in recordings if r['num_seizures'] == 0]

print(f"Recordings with seizures: {len(with_seizures)}")
print(f"Recordings without seizures (normal): {len(without_seizures)}")

# Load example normal and seizure recordings
print("\nLoading sample signals...")

# Load a normal recording (no seizures)
if without_seizures:
    normal_rec = without_seizures[0]
    try:
        normal_signals = loader.load_subject_session(
            normal_rec['subject_id'],
            normal_rec['session_id'],
            normal_rec['run_id']
        )
        normal_ecg, normal_fs = normal_signals.get('ecg', (None, None))
        print(f"✓ Loaded normal ECG: shape={normal_ecg.shape if normal_ecg is not None else None}")
    except Exception as e:
        logger.warning(f"Could not load normal signal: {e}")
        normal_ecg = None

# Load a seizure recording
if with_seizures:
    seizure_rec = with_seizures[0]
    try:
        seizure_signals = loader.load_subject_session(
            seizure_rec['subject_id'],
            seizure_rec['session_id'],
            seizure_rec['run_id']
        )
        seizure_ecg, seizure_fs = seizure_signals.get('ecg', (None, None))
        print(f"✓ Loaded seizure ECG: shape={seizure_ecg.shape if seizure_ecg is not None else None}")
    except Exception as e:
        logger.warning(f"Could not load seizure signal: {e}")
        seizure_ecg = None

In [ ]:
# Create visualizer
viz = SignalVisualizer(save_dir="../visualizations", upload_to_wandb=False)

# Plot signal comparison if both available
if normal_ecg is not None and seizure_ecg is not None:
    # Extract shorter segments for clarity (30 seconds)
    n_samples = min(int(normal_fs * 30), normal_ecg.shape[1], seizure_ecg.shape[1])
    
    normal_segment = normal_ecg[0, :n_samples]
    seizure_segment = seizure_ecg[0, :n_samples]
    
    # Plot comparison
    fig = viz.plot_signal_comparison(
        normal_segment,
        seizure_segment,
        sampling_rate=normal_fs,
        signal_name="ECG"
    )
    plt.show()
    viz.save_figure(fig, "signal_comparison_normal_vs_seizure")
    print("✓ Signal comparison saved")
else:
    print("⚠ Could not load both normal and seizure signals for comparison")

## 3. Simulate Model Predictions

Create example predictions and visualize detection performance.

In [ ]:
# Generate example countdown predictions
np.random.seed(42)

# Scenario 1: Perfect prediction
n_timesteps = 1200  # 4 minutes at 5 Hz

# True countdown (starts at -1 inter-ictal, transitions to 10 min pre-ictal, counts down to seizure)
true_countdown = np.concatenate([
    np.full(200, -1),                        # 40 seconds inter-ictal
    np.arange(10, -0.01, -10/600),          # 10 min countdown (120 samples per minute)
    np.full(n_timesteps - 800, -1)          # Post-seizure inter-ictal
])[:n_timesteps]

# Predicted countdown (with small noise)
pred_countdown = true_countdown + 0.3 * np.random.randn(n_timesteps)
pred_countdown = np.clip(pred_countdown, -1, 10)

# Pre-ictal classification
pre_ictal_true = (true_countdown >= 0).astype(float)
pre_ictal_pred = pre_ictal_true + 0.1 * np.random.randn(n_timesteps)
pre_ictal_pred = np.clip(pre_ictal_pred, 0, 1)

# Plot countdown prediction
fig = viz.plot_countdown_prediction(
    true_countdown,
    pred_countdown,
    pre_ictal_true,
    pre_ictal_pred,
    sampling_rate=5
)
plt.show()
viz.save_figure(fig, "countdown_prediction_example")
print("✓ Countdown prediction visualization saved")

## 4. Detection Error Analysis

Analyze and visualize prediction errors and performance metrics.

In [ ]:
# Calculate error metrics
preictal_mask = true_countdown >= 0
mae = np.mean(np.abs(pred_countdown[preictal_mask] - true_countdown[preictal_mask]))
medae = np.median(np.abs(pred_countdown[preictal_mask] - true_countdown[preictal_mask]))
rmse = np.sqrt(np.mean((pred_countdown[preictal_mask] - true_countdown[preictal_mask]) ** 2))

print(f"Error Metrics (pre-ictal only):")
print(f"  MAE:   {mae:.3f} minutes")
print(f"  MEDAE: {medae:.3f} minutes")
print(f"  RMSE:  {rmse:.3f} minutes")

# Detection sensitivity at different lead times
lead_times = [3, 5, 10]  # minutes
print(f"\nDetection Sensitivity:")
for lead_time in lead_times:
    detected = np.sum((pred_countdown[preictal_mask] >= lead_time) & (true_countdown[preictal_mask] >= lead_time))
    available = np.sum(true_countdown[preictal_mask] >= lead_time)
    sensitivity = detected / available if available > 0 else 0
    print(f"  @ {lead_time} min lead time: {sensitivity:.1%} ({detected}/{available})")

# Plot error distribution
fig = viz.plot_error_distribution(true_countdown[preictal_mask], pred_countdown[preictal_mask])
plt.show()
viz.save_figure(fig, "prediction_error_distribution")
print("\n✓ Error distribution visualization saved")

## 5. Weights & Biases Integration

Log visualizations and metrics to Weights & Biases for experiment tracking.

In [ ]:
if HAS_WANDB:
    # Initialize wandb run
    run = wandb.init(
        project="epilepsee-ai",
        name="signal_visualization_analysis",
        tags=["visualization", "signal-analysis", "notebook"],
    )
    
    # Log metrics
    wandb.log({
        "dataset/total_seizures": total_seizures,
        "dataset/total_recordings": len(recordings),
        "dataset/total_subjects": len(loader.get_subjects()),
        "dataset/recordings_with_seizures": len(with_seizures),
        "dataset/recordings_without_seizures": len(without_seizures),
        
        "metrics/mae": mae,
        "metrics/medae": medae,
        "metrics/rmse": rmse,
        "metrics/sensitivity_3min": np.sum((pred_countdown[preictal_mask] >= 3) & (true_countdown[preictal_mask] >= 3)) / np.sum(true_countdown[preictal_mask] >= 3),
        "metrics/sensitivity_5min": np.sum((pred_countdown[preictal_mask] >= 5) & (true_countdown[preictal_mask] >= 5)) / np.sum(true_countdown[preictal_mask] >= 5),
        "metrics/sensitivity_10min": np.sum((pred_countdown[preictal_mask] >= 10) & (true_countdown[preictal_mask] >= 10)) / np.sum(true_countdown[preictal_mask] >= 10),
    })
    
    print("✓ Metrics logged to wandb")
    print(f"View run: {run.get_url()}")
    
    wandb.finish()
else:
    print("⚠ wandb not available - skipping logging")